# 10 · Atypical example level: search for matched models
### Machine Unlearning is Extinction, Not Forgetting

**Why this notebook.** In notebook 09 (run #12) no model met the matching rule. RL and SalUn kept their test
accuracy but stopped at forget accuracy 0.16 / 0.19, against 0.034 for the retrained model. The calibration showed
both still improving at the edge of the grid (RL at lr 0.02: 0.274 → 0.208 → 0.154 after 1 / 3 / 5 epochs), so longer
runs may reach the retrained level. NegGrad+ reached it, but only by losing 6-16 test points; a gentler, longer
schedule may keep more utility. Matched models remove the main objection to the example-level relapse result
("the models only forgot partly").

**What runs.** Three method *variants* with their own grids. They use exactly the same unlearning code as their base
methods, with their own job keys and checkpoints, so nothing from notebook 09 is overwritten:

| Variant | Base | Grid | Calibration jobs |
|---|---|---|---:|
| `RL-long` | RL | lr 0.02 × 10 / 15 / 20 epochs; lr 0.05 × 10 / 20 epochs | 5 |
| `SalUn-long` | SalUn | same | 5 |
| `NegGrad+-long` | NegGrad+ | α 0.95 / 0.97 × lr 0.01 / 0.02, 6 epochs | 4 |

| Stage | What | Jobs | Time on T4 x2 |
|---|---|---|---|
| **B** calibrate | the grids above on the pilot forget set | 14 | ~25 min |
| **C** unlearn | 3 variants × 9 forget sets (+ 3 pilot), up to 3 candidates each until matched | 30 | ~1-2 h |
| **D** relapse tests | identical to notebook 09, for the new models only | 27 | ~35 min |

The retrained models, the forget sets and the relapse tests of the original and retrained models are reused from
`ext-09-atypical`. **ext_utils 1.5** also ranks unmatched candidates by how far they miss the matching rule (forget
distance beyond ±5 points plus test shortfall beyond 3 points), not by forget distance alone.

**Analysis rule (fixed before the data):** per (method, seed, forget set), a matched model from this notebook replaces
notebook 09's unmatched one; otherwise notebook 09's model stays. The matched subset is tested on its own (XH1, B1).

## Kaggle settings for this notebook

| Setting | Value |
|---|---|
| **Accelerator** | **GPU T4 x2** |
| **Internet** | On (only a fallback for the C-scores) |
| **Environment** | Latest container image |

**Inputs to attach** (*Add Input* → *Your Datasets*):

| Input | Why |
|---|---|
| `ext-code` (**version 1.5**: `ext_utils.py` and `cifar100-cscores-orig-order.npz`) | method variants, new candidate ranking |
| CIFAR-100 (`fedesoriano/cifar100`) | data |
| `ext-01-originals` | starting point of every unlearning run |
| `ext-09-atypical` | the 10 retrained models (targets) and notebook 09's results; **required** |

Do **not** attach `ext-02` / `ext-03` (not needed). **How to run:** *Save Version → Save & Run All (Commit)*. If it
stops early, attach this notebook's own output as well and run again: finished jobs are skipped.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

Identical to notebook 09 except the methods (the three variants), their grids, and `max_tries = 3`.

In [ ]:
GRID = {
    "RL-long":       [{"lr": lr, "epochs": e} for lr, e in [(0.02, 10), (0.02, 15), (0.02, 20), (0.05, 10), (0.05, 20)]],
    "SalUn-long":    [{"lr": lr, "epochs": e, "mask_ratio": 0.5} for lr, e in [(0.02, 10), (0.02, 15), (0.02, 20), (0.05, 10), (0.05, 20)]],
    "NegGrad+-long": [{"lr": lr, "alpha": a, "epochs": 6} for a in (0.95, 0.97) for lr in (0.01, 0.02)],
}
CFG = U.make_cfg(
    smoke=False,
    seeds=[0, 1, 2],
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],
    pilot_class="lamp",
    norm="bn",
    width=64, epochs=40, batch_size=256, lr=0.1, pct_start=0.25,
    momentum=0.9, nesterov=True, weight_decay=5e-4, label_smoothing=0.0,
    amp=True, channels_last=True, cudnn_benchmark=True, eval_batch_size=1024,
    ckpt_fp16=True, retain_eval_subset=5000,
    methods=["RL-long", "SalUn-long", "NegGrad+-long"],
    forget_batch_size=64, clip_grad=5.0, match_retain_gap=0.03, max_tries=3,
    aba_classes=["maple_tree", "tiger", "rose", "shark", "bus"], aba_context="gray",
    mctx_classes=["maple_tree", "tiger", "rose", "shark", "bus"], mctx_contexts=["clean", "gray", "blur"],
    recovery_bn_modes=["update", "frozen"], bn_recal_sizes=[10, 100, 1000, 10000],
    sr_epochs=5, sr_lr=0.01, sr_noise=[0.005, 0.01, 0.02, 0.05], sr_noise_reps=3,
    sr_quant_bits=[8, 6, 4], sr_prune=[0.2, 0.5, 0.7], ctx_synthetic=["gray", "blur", "noise", "contrast"],
    sv_steps=100, sv_lr=0.01, sv_replay_bs=128, sv_eval_steps=[0, 1, 2, 5, 10, 20, 50, 100], collapse_frac=0.5,
    # example-level arm, atypical design
    xl_design="atypical", xl_pool_frac=0.05, xl_n=500,
    xl_draws=[0, 1, 2], xl_pilot_draw=9, xl_match_gap=0.05, xl_sv_shots=[10, 100], xl_calib_grid=GRID,
    time_budget_h=11.0, n_workers=None, reuse_ckpt=True, raise_on_error=True, poll_seconds=60,
)
RUN_B, RUN_C, RUN_D = True, True, True
U.check_experiment_config(CFG)
assert U.VERSION >= "1.5", "Attach ext-code version 1.5."

## Check the inputs before spending GPU

Every retrained model from notebook 09 must be found (they are the matching targets), and the forget sets must be the
same as notebook 09's (same C-scores, same draws).

In [ ]:
import numpy as np, pandas as pd
_D = U.load_data(CFG); _G = U.gpu_data(_D, CFG, "cpu")
_sc = U.cscores(CFG, _G)
_t = U.xl_target(_G, CFG, CFG["seeds"][0], CFG["xl_pilot_draw"])
print(f"C-scores: {len(_sc)} images, cut-off {np.sort(_sc)[int(CFG['xl_pool_frac'] * len(_sc)) - 1]:.3f} (notebook 09: 0.025); "
      f"pilot forget set: {len(_t.fidx)} images, C-score mean {_sc[_t.fidx.numpy()].mean():.3f} (notebook 09: 0.010)")
del _D, _G
units = U._xl_units(CFG, True) + U._xl_units(CFG, False)
missing = [(s, d) for s, d in units if U.find_file(U._xck("retrain", CFG, d, s)) is None]
assert not missing, f"retrained models not found for {missing}: attach ext-09-atypical"
old = U.load_df("xl_unlearn", CFG)
print(f"retrained models found: {len(units)}/{len(units)}; notebook 09 unlearning rows found: {len(old)}")

## Stage B: calibration on the pilot forget set

In [ ]:
if RUN_B:
    jobs = U.jobs_xl_calibrate(CFG)
    print(f"{len(jobs)} calibration jobs")
    U.launch("xl_calibrate", jobs, CFG)
    xl_candidates, xl_table = U.xl_select_candidates(CFG)
    display(xl_table.round(4))
xl_candidates = U.load_xl_candidates(CFG)
for m, c in xl_candidates.items():
    print(f"{m:14s} tries in order: {c}")

**How to read it.** The pilot target is forget accuracy ≤ 0.076 (retrained 0.026 + 5 points) with test accuracy
≥ 0.720. Rows with `matched = True` are what we want; if none, the closest candidates still run.

## Stage C: unlearning

In [ ]:
if RUN_C:
    U.launch("xl_unlearn", U.jobs_xl_unlearn(CFG, xl_candidates, pilot=True) + U.jobs_xl_unlearn(CFG, xl_candidates), CFG)

## Stage D: relapse tests (new models only)

In [ ]:
if RUN_D:
    jobs = U.jobs_xl_extinction(CFG)
    print(f"{len(jobs)} models in the list (the original and retrained models were tested in notebook 09 and are skipped)")
    U.launch("xl_extinction", jobs, CFG)

## Quick look: new variants next to notebook 09's models

In [ ]:
def _atyp(d): return d[d["design"] == "atypical"] if len(d) and "design" in d else d.iloc[0:0]
xu = _atyp(U.load_df("xl_unlearn", CFG))
if len(xu):
    main = xu[xu["draw"] != CFG["xl_pilot_draw"]]
    print("matched rate and accuracies (means over 9 forget sets; retrained forget 0.034, test 0.750):")
    display(main.groupby("method")[["matched", "forget_acc", "ref_forget_acc", "test_acc", "ref_test_acc"]].mean().round(4))
    display(main[main["method"].isin(CFG["methods"])][["method", "seed", "draw", "matched", "forget_acc", "ref_forget_acc", "test_acc", "hp"]]
            .sort_values(["method", "seed", "draw"]).round(3))
xe = _atyp(U.load_df("xl_extinction", CFG))
if len(xe):
    ids = ["kind", "method", "seed", "draw"]
    base = xe[xe["test"] == "base"][ids + ["forget_acc"]].rename(columns={"forget_acc": "before"})
    rf = xe[(xe["test"] == "spontaneous") & (xe["proxy"] == "retain_ft") & (xe["bn"] == "update")
            & np.isclose(pd.to_numeric(xe["level"], errors="coerce"), CFG["sr_epochs"])]
    rf = rf.merge(base, on=ids).assign(recovery=lambda d: d["forget_acc"] - d["before"])
    if len(xu):
        rf = rf.merge(xu[["method", "seed", "draw", "matched"]].rename(columns={"matched": "is_matched"}), on=["method", "seed", "draw"], how="left")
        rf["is_matched"] = rf["is_matched"].astype("boolean").fillna(False).astype(bool)
    print("retain fine-tuning (5 epochs, BN update): forget accuracy after minus before:")
    display(rf.groupby(["method", "is_matched"] if "is_matched" in rf else ["method"])[["before", "forget_acc", "recovery"]]
            .agg(["mean", "count"]).round(4))

## Next step

Save a version and save its output as the dataset **`ext-10-matched-search`**. Then send the log (and screenshots of the
calibration table and the quick look). Notebook 08 will get an update that merges the variants with notebook 09's models
by the rule above.

| Output name | Contains | Attached by |
|---|---|---|
| `ext-10-matched-search` | variant `xl_calibrate` / `xl_unlearn` / `xl_extinction` results, 30 new checkpoints, plus copies of the notebook 09 rows it used | notebook 08 |